# 信贷风控规则挖掘 + 评分卡建模完整流程
流程: 数据加载 → IV计算 → LightGBM建模 → 规则提取/评估 → PSI稳定性 → 风险趋势 → 相关性筛选 → 逻辑回归评分卡

In [ ]:
# ============================================================
# 第一部分：导入依赖
# ============================================================
import numpy as np
import pandas as pd
import os, re, time, gc, copy, logging, warnings, multiprocessing, joblib
from datetime import datetime
from collections import Counter
from typing import List, Dict, Union

# 机器学习
from sklearn import tree, metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, roc_curve, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import KBinsDiscretizer
import lightgbm as lgb

# 分箱与IV
from optbinning import OptimalBinning, Scorecard, BinningProcess
from scipy.stats import spearmanr, kruskal, chi2_contingency, kendalltau

# 并行与进度
from joblib import Parallel, delayed
from tqdm import tqdm

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from matplotlib.backends.backend_pdf import PdfPages

# toad
import toad
from toad.plot import bin_plot
from toad.transform import Combiner

# ODPS
import za_mlplatform_sdk as mlp
from odps.df import DataFrame

# 配置
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logging.getLogger('optbinning').setLevel(logging.WARNING)
warnings.filterwarnings("ignore")

# 中文字体
font_path = '/root/fonts/simhei.ttf'
font_prop = font_manager.FontProperties(fname=font_path)
font_manager.fontManager.addfont(font_path)
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# ============================================================
# 第二部分：数据加载
# ============================================================
odps = mlp.get_odps_instance(data_id="data13a88922e07511f089a70242ac850002")

def read_data_from_odps(odpstablename):
    table = DataFrame(odps.get_table(odpstablename))
    n_process = multiprocessing.cpu_count()
    print(f'n_process: {n_process}')
    dt1 = datetime.now()
    print(f'开始: {dt1.strftime("%Y-%m-%d %H:%M:%S")}')
    df = table.to_pandas(n_process=n_process)
    dt2 = datetime.now()
    print(f'结束: {dt2.strftime("%Y-%m-%d %H:%M:%S")} | 耗时: {(dt2-dt1).seconds/60:.1f}分钟 | shape: {df.shape}')
    return df

df = read_data_from_odps('yy_apply_kb_zj_05_nobr_xf_pass_rule_01_202506_0617')
df.head(3)

In [ ]:
# ============================================================
# 第三部分：IV计算
# ============================================================
def calculate_iv_single(col_name, data):
    """快速计算单变量IV（拉普拉斯平滑）"""
    try:
        if len(data) < 50:
            return None
        feature_series = data.iloc[:, 0]
        target_series = data.iloc[:, 1]

        if pd.api.types.is_numeric_dtype(feature_series):
            n_bins = min(10, len(data) // 50)
            try:
                bins = pd.qcut(feature_series, q=n_bins, duplicates='drop', labels=False)
            except:
                bins = pd.cut(feature_series, bins=n_bins, labels=False, duplicates='drop')
            dtype = 'numeric'
        else:
            bins = feature_series.astype('category').cat.codes
            dtype = 'categorical'

        df_temp = pd.DataFrame({'target': target_series.values, 'bins': bins.values}).dropna()
        if len(df_temp) < 2:
            return None

        grouped = df_temp.groupby('bins')['target'].agg(['count', 'sum'])
        grouped['non_events'] = grouped['count'] - grouped['sum']
        total_events = grouped['sum'].sum()
        total_non_events = grouped['non_events'].sum()
        if total_events == 0 or total_non_events == 0:
            return None

        grouped['event_pct'] = (grouped['sum'] + 0.5) / (total_events + 1)
        grouped['non_event_pct'] = (grouped['non_events'] + 0.5) / (total_non_events + 1)
        grouped['woe'] = np.log(grouped['event_pct'] / grouped['non_event_pct'])
        iv = ((grouped['event_pct'] - grouped['non_event_pct']) * grouped['woe']).sum()
        return {'variable': col_name, 'iv': iv, 'dtype': dtype}
    except:
        return None


def compute_iv_fast(df, target_col='dob4_ever10_flg'):
    """并行IV计算"""
    cols = [c for c in df.columns if c != target_col]
    missing_rates = df[cols].isnull().mean()
    nunique_vals = df[cols].nunique()
    valid_cols = [c for c in cols if missing_rates[c] <= 0.8 and nunique_vals[c] > 1]

    results = Parallel(n_jobs=-1)(
        delayed(calculate_iv_single)(col, df[[col, target_col]].dropna())
        for col in tqdm(valid_cols, desc="IV计算")
    )
    iv_df = pd.DataFrame([r for r in results if r is not None])
    if not iv_df.empty:
        iv_df = iv_df.sort_values('iv', ascending=False).reset_index(drop=True)
        iv_df['strength'] = np.select(
            [iv_df['iv'] <= 0.02, iv_df['iv'] <= 0.1, iv_df['iv'] <= 0.3, iv_df['iv'] > 0.3],
            ['无预测力', '弱', '中等', '强'], default='无预测力'
        )
    return iv_df

In [ ]:
# ============================================================
# 第四部分：变量分析（PDF输出）
# ============================================================
def var_analysis(raw_data, variables, target_variable, var_month, out_pdf):
    months = sorted(raw_data[var_month].unique())
    nmonth = len(months) + 1
    with PdfPages(out_pdf) as pdf:
        for var in tqdm(variables, desc="变量分析"):
            try:
                fig, axes = plt.subplots(1, nmonth, figsize=(20, 5))
                for i, month in enumerate(months):
                    monthly_data = raw_data[raw_data[var_month] == month].copy()
                    monthly_data['bin'] = pd.qcut(monthly_data[var], q=10, duplicates='drop')
                    grouped = monthly_data.groupby('bin')[target_variable].agg(['count', 'sum'])
                    grouped.columns = ['total', 'bad']
                    grouped['bad_rate'] = grouped['bad'] / grouped['total']
                    grouped['good'] = grouped['total'] - grouped['bad']
                    total_good, total_bad = grouped['good'].sum(), grouped['bad'].sum()
                    grouped['woe'] = np.log((grouped['good']/total_good) / (grouped['bad']/total_bad))
                    grouped['iv'] = ((grouped['good']/total_good) - (grouped['bad']/total_bad)) * grouped['woe']
                    iv_value = grouped['iv'].sum()

                    ax1 = axes[i]
                    ax1.bar(grouped.index.astype(str), grouped['total'], color='skyblue')
                    ax1.set_title(f'{var} - {month} (IV={iv_value:.4f})', fontsize=10, fontweight='bold')
                    ax1.set_xticklabels(grouped.index.astype(str), rotation=45, ha='right')
                    ax2 = ax1.twinx()
                    ax2.plot(grouped.index.astype(str), grouped['bad_rate'], color='red', marker='o')
                    for j, br in enumerate(grouped['bad_rate']):
                        ax2.text(j, br + 0.002, f'{br*100:.2f}%', ha='center', fontsize=8)

                # 总图
                tmp = raw_data.copy()
                tmp['bin'] = pd.qcut(tmp[var], q=10, duplicates='drop')
                grouped = tmp.groupby('bin')[target_variable].agg(['count', 'sum'])
                grouped.columns = ['total', 'bad']
                grouped['bad_rate'] = grouped['bad'] / grouped['total']
                grouped['good'] = grouped['total'] - grouped['bad']
                total_good, total_bad = grouped['good'].sum(), grouped['bad'].sum()
                grouped['woe'] = np.log((grouped['good']/total_good) / (grouped['bad']/total_bad))
                grouped['iv'] = ((grouped['good']/total_good) - (grouped['bad']/total_bad)) * grouped['woe']
                iv_value = grouped['iv'].sum()
                ax1 = axes[-1]
                ax1.bar(grouped.index.astype(str), grouped['total'], color='skyblue')
                ax1.set_title(f'{var} - 总图 (IV={iv_value:.4f})', fontsize=10, fontweight='bold')
                ax1.set_xticklabels(grouped.index.astype(str), rotation=45, ha='right')
                ax2 = ax1.twinx()
                ax2.plot(grouped.index.astype(str), grouped['bad_rate'], color='red', marker='o')
                for j, br in enumerate(grouped['bad_rate']):
                    ax2.text(j, br + 0.002, f'{br*100:.2f}%', ha='center', fontsize=8)
                fig.tight_layout()
                pdf.savefig(fig)
                plt.close()
            except Exception as e:
                print(f"Error: {var} - {e}")

In [ ]:
# ============================================================
# 第五部分：模型评估工具
# ============================================================
def plot_matrix_report(y_label, y_pred):
    matrix_array = metrics.confusion_matrix(y_label, y_pred)
    plt.matshow(matrix_array, cmap=plt.cm.summer_r)
    plt.colorbar()
    for x in range(len(matrix_array)):
        for y in range(len(matrix_array)):
            plt.annotate(matrix_array[x, y], xy=(x, y), ha='center', va='center')
    plt.xlabel('True label')
    plt.ylabel('Predict label')
    print(metrics.classification_report(y_label, y_pred))
    plt.show()

def PlotKS(preds, labels, n=10000, asc=0):
    ksds = pd.DataFrame({'bad': labels, 'pred': preds})
    ksds['good'] = 1 - ksds.bad
    sort_asc = [True, True] if asc == 1 else [False, True]
    ksds1 = ksds.sort_values(by=['pred', 'bad'], ascending=sort_asc).reset_index(drop=True)
    ksds1['cumsum_good1'] = ksds1.good.cumsum() / ksds1.good.sum()
    ksds1['cumsum_bad1'] = ksds1.bad.cumsum() / ksds1.bad.sum()
    sort_asc2 = [True, False] if asc == 1 else [False, False]
    ksds2 = ksds.sort_values(by=['pred', 'bad'], ascending=sort_asc2).reset_index(drop=True)
    ksds2['cumsum_good2'] = ksds2.good.cumsum() / ksds2.good.sum()
    ksds2['cumsum_bad2'] = ksds2.bad.cumsum() / ksds2.bad.sum()

    ksds_final = pd.DataFrame({
        'cumsum_good': (ksds1['cumsum_good1'] + ksds2['cumsum_good2']) / 2,
        'cumsum_bad': (ksds1['cumsum_bad1'] + ksds2['cumsum_bad2']) / 2
    })
    ksds_final['ks'] = ksds_final['cumsum_bad'] - ksds_final['cumsum_good']
    ksds_final['tile'] = np.arange(1, len(ksds_final)+1) / len(ksds_final)

    qe = np.linspace(0, 1, n+1)[1:]
    idx = np.ceil(pd.Series(ksds_final.index).quantile(q=qe)).astype(int).values
    ksds_final = ksds_final.loc[idx]
    ksds_final = pd.concat([pd.DataFrame([[0,0,0,0]], columns=ksds_final.columns), ksds_final]).reset_index(drop=True)

    ks_value = ksds_final.ks.max()
    ks_pop = ksds_final.tile[ksds_final.ks.idxmax()]
    print(f'KS={ks_value:.4f} at Pop={ks_pop:.4f}')

    plt.figure()
    plt.plot(ksds_final.tile, ksds_final.cumsum_good, 'b-', lw=2, label='cum_good')
    plt.plot(ksds_final.tile, ksds_final.cumsum_bad, 'r-', lw=2, label='cum_bad')
    plt.plot(ksds_final.tile, ksds_final.ks, 'g-', lw=2, label='ks')
    plt.axvline(ks_pop, color='gray', ls='--')
    plt.title(f'KS={ks_value:.4f} at Pop={ks_pop:.4f}', fontsize=15)
    plt.legend()
    plt.show()

def PlotROC(preds, labels):
    fpr, tpr, _ = roc_curve(labels, preds, pos_label=1)
    auc_score = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC={auc_score:.5f}')
    plt.plot([0,1],[0,1], '--', color=(0.6,0.6,0.6))
    plt.title('ROC Curve')
    plt.legend()
    plt.show()

def m_plot(model, X, y, threshold=0.5, n=1000, asc=0):
    y_predicted = model.predict(X)
    y_pred = [int(v > threshold) for v in y_predicted]
    plot_matrix_report(y, y_pred)
    PlotKS(y_predicted, y, n=n, asc=asc)
    PlotROC(y_predicted, y)

In [ ]:
# ============================================================
# 第六部分：排除字段配置
# ============================================================
to_drop = [
    'channel_type_final', 'product_code', 'loan_apply_no', 'apply_month',
    'credit_apply_date', 'credit_amount',
    'is_perform_fpb1', 'is_perform_fpb5', 'is_perform_fpb10', 'is_perform_fpb31',
    'is_perform_dob2_ever10', 'is_perform_dob3_ever10', 'is_perform_dob4_ever10',
    'is_perform_dob5_ever10', 'is_perform_dob6_ever10',
    'fpd1_flag', 'fpd5_flag', 'fpd10_flag', 'fpd31_flag',
    'dob2_ever10_flg', 'dob3_ever10_flg', 'dob4_ever10_flg', 'dob5_ever10_flg', 'dob6_ever10_flg',
    'dob2_ever30_flg', 'dob3_ever30_flg', 'dob4_ever30_flg', 'dob5_ever30_flg', 'dob6_ever30_flg',
    'fpd1_amt', 'fpd5_amt', 'fpd10_amt', 'fpd31_amt',
    'dob2_ever10_amt', 'dob3_ever10_amt', 'dob4_ever10_amt', 'dob5_ever10_amt', 'dob6_ever10_amt',
    'dob2_ever30_amt', 'dob3_ever30_amt', 'dob4_ever30_amt', 'dob5_ever30_amt', 'dob6_ever30_amt',
    'rule_new', 'hit_rule_new', 'hit_rule_new2', 'now_credit_amount', 'trans_withdraw_amount',
    'dxm_general_prea_consc_v2_score', 'user_id',
    'pd_ylzc_shouyufen_upsd001_score', 'pd_zj_25088_score',
    'td_i_cnt_node_dist2_loan_all_all',
    'tcxy_creditpro_v1_repaysuc_cnt_360d_ratio_by_trans_cnt_360d',
    'pudao_jig_credit_cs1_score', 'pd_txty_hrate7_score',
    'pd_baiduyun_duyifen_a_02_8', 'pd_jd_xuanyuan_plus',
    'pd_gd_za_cust_score_v1', 'pd_blz_tl_yh_v1', 'pd_dxm_general_prea_v10',
    'txty_total_xe4_5_score', 'jbx_scoree_score_e',
    'pd_ylzc_shouyufen_upsd002_score',
    'credit_risk_score_x2_bys', 'credit_risk_score_x3_bys',
    'tcxy_applypro_v1_apply_model_score_high', 'pd_zj_17178_score',
]

In [ ]:
# ============================================================
# 第七部分：计算IV
# ============================================================
raw_data = df[df['is_perform_dob4_ever10'] > 0].copy()
iv_result = compute_iv_fast(
    raw_data.drop(columns=[c for c in to_drop if c in raw_data.columns], errors='ignore'),
    'dob4_ever10_flg'
)
iv_result.head(20)

In [ ]:
# ============================================================
# 第八部分：准备建模数据
# ============================================================
dftmp = df[df['is_perform_dob4_ever10'] > 0].copy()
scoreVar = [x for x in dftmp.columns if 'td_' in x or 'score' in x or 'bh_' in x]

train_data_x = dftmp[list(set(scoreVar) - set(to_drop))].copy()
train_data_y = dftmp['dob4_ever10_flg']

# 类型转换
for col in train_data_x.select_dtypes('object').columns:
    try:
        train_data_x[col] = pd.to_numeric(train_data_x[col])
    except ValueError:
        train_data_x[col] = train_data_x[col].astype('category')

print(f"特征数: {train_data_x.shape[1]} | 样本: {train_data_x.shape[0]}")

In [ ]:
# ============================================================
# 第九部分：LightGBM两阶段训练
# ============================================================
X_train, X_val, y_train, y_val = train_test_split(
    train_data_x, train_data_y, test_size=0.3, random_state=42, stratify=train_data_y
)
train_set = lgb.Dataset(X_train, y_train, free_raw_data=False)
val_set = lgb.Dataset(X_val, y_val, reference=train_set, free_raw_data=False)

base_params = {
    'boosting_type': 'gbdt', 'objective': 'binary',
    'metric': ['auc', 'binary_error'],
    'num_leaves': 31, 'max_depth': 5, 'min_data_in_leaf': 1000,
    'learning_rate': 0.1, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'scale_pos_weight': len(y_train[y_train==0]) / len(y_train[y_train==1]),
    'verbose': -1
}

# Stage 1: GBDT
print("=== Stage 1: GBDT ===")
lgbmodel = lgb.train(
    base_params, train_set, num_boost_round=200,
    valid_sets=[train_set, val_set], valid_names=['train', 'valid'],
    callbacks=[lgb.log_evaluation(10), lgb.early_stopping(50)]
)

# Stage 2: DART
print("\n=== Stage 2: DART ===")
stage2_params = {**base_params, 'learning_rate': 0.02, 'boosting_type': 'dart'}
lgbmodel = lgb.train(
    stage2_params, train_set, num_boost_round=300,
    valid_sets=[train_set, val_set], valid_names=['train', 'valid'],
    callbacks=[lgb.log_evaluation(10), lgb.early_stopping(50)]
)

print(f"\n验证集AUC: {roc_auc_score(y_val, lgbmodel.predict(X_val)):.4f}")

In [ ]:
m_plot(lgbmodel, X_val, y_val, threshold=0.5)

In [ ]:
importance_df = pd.DataFrame({
    'feature': lgbmodel.feature_name(),
    'importance': lgbmodel.feature_importance()
}).sort_values('importance', ascending=False).reset_index(drop=True)
importance_df.head(10)

In [ ]:
# ============================================================
# 第十部分：规则提取（优化版）
# ============================================================
def extract_decision_paths_fast(model, feature_names, sample_data, max_depth=4):
    leaf_ids = model.predict(sample_data, pred_leaf=True)
    tree_dicts = model.dump_model()['tree_info']

    tree_paths = []
    for tree_info in tree_dicts:
        node_paths = {}
        stack = [(tree_info['tree_structure'], [], 0)]
        while stack:
            node, path, depth = stack.pop()
            if depth >= max_depth or 'split_feature' not in node:
                if 'leaf_index' in node:
                    node_paths[node['leaf_index']] = ' AND '.join(path)
                continue
            feat_name = feature_names[node['split_feature']]
            thresh = node['threshold']
            fmt = f"{thresh:.4f}" if isinstance(thresh, float) else str(thresh)
            if 'left_child' in node:
                stack.append((node['left_child'], path + [f"{feat_name}<={fmt}"], depth+1))
            if 'right_child' in node:
                stack.append((node['right_child'], path + [f"{feat_name}>{fmt}"], depth+1))
        tree_paths.append(node_paths)

    rules = []
    for tree_idx in range(leaf_ids.shape[1]):
        path_map = tree_paths[tree_idx]
        for leaf_id in leaf_ids[:, tree_idx]:
            r = path_map.get(leaf_id)
            if r:
                rules.append(r)
    return pd.Series(rules).value_counts()

top_rules = extract_decision_paths_fast(lgbmodel, train_data_x.columns.tolist(), train_data_x, max_depth=4)
print(f"规则总数: {len(top_rules)}")

In [ ]:
# ============================================================
# 第十一部分：规则评估（优化版：正则+numpy+并行）
# ============================================================
_CONDITION_RE = re.compile(r'^(.+?)\s*(<=|>=|!=|==|>|<)\s*(.+)$')

def parse_conditions(rule_str):
    normalized = rule_str.replace('≤', '<=').replace('≥', '>=').strip()
    return [(m.group(1).strip(), m.group(2), m.group(3).strip())
            for cond in normalized.split(' AND ')
            if (m := _CONDITION_RE.match(cond.strip()))]

def should_exclude_rule(rule_str, good_vars_set, bad_vars_set):
    for var, op, _ in parse_conditions(rule_str):
        if var in good_vars_set and op in ('>', '>='):
            return True
        if var in bad_vars_set and op in ('<', '<='):
            return True
    return False

def _eval_single_rule(rule, df_values, col_index, label_arr, n_total, total_bad_rate, min_cov, max_cov, min_bads):
    conditions = parse_conditions(rule)
    if not conditions:
        return None
    mask = np.ones(n_total, dtype=bool)
    for var, op, val_str in conditions:
        idx = col_index.get(var)
        if idx is None:
            return None
        col = df_values[:, idx]
        try:
            val = float(val_str)
        except ValueError:
            return None
        if op == '<=': mask &= col <= val
        elif op == '>=': mask &= col >= val
        elif op == '>': mask &= col > val
        elif op == '<': mask &= col < val
        elif op == '==': mask &= col == val
        elif op == '!=': mask &= col != val
        if not mask.any():
            return None
    hit = mask.sum()
    coverage = hit / n_total
    if coverage < min_cov or coverage > max_cov:
        return None
    bads = label_arr[mask].sum()
    if bads < min_bads:
        return None
    bad_rate = bads / hit
    return {'原始规则': rule, '命中样本': int(hit), '坏样本数': int(bads),
            '覆盖率': f"{coverage:.2%}", '坏账率': f"{bad_rate:.2%}",
            '提升度': f"{bad_rate/total_bad_rate:.2f}", 'bad_rate': bad_rate}

def evaluate_rules(df, rules, label_col='dob4_ever10_flg',
                   good_vars=None, bad_vars=None,
                   min_coverage=0.05, max_coverage=0.50, min_bads=50, n_jobs=-1):
    # 自动识别
    auto_good = [c for c in df.columns if 'model' in c.lower() and 'score' in c.lower()]
    good_set = set((good_vars or []) + auto_good)
    risk_pat = re.compile(r'(num|cnt|org|amt)', re.IGNORECASE)
    auto_bad = [c for c in df.columns if risk_pat.search(c) and c not in good_set]
    bad_set = set((bad_vars or []) + auto_bad)
    print(f"good_vars: {len(good_set)} | bad_vars: {len(bad_set)}")

    valid_rules = [r for r in rules if not should_exclude_rule(r, good_set, bad_set)]
    print(f"过滤: {len(rules)} → {len(valid_rules)}")

    all_vars = set()
    for r in valid_rules:
        for var, _, _ in parse_conditions(r):
            all_vars.add(var)
    used_cols = [c for c in df.columns if c in all_vars]
    col_index = {c: i for i, c in enumerate(used_cols)}

    df_num = df[used_cols].apply(pd.to_numeric, errors='coerce')
    df_values = df_num.values.astype(np.float64)
    label_arr = df[label_col].values.astype(np.float64)
    n_total = len(df)
    total_bad_rate = label_arr.mean()
    print(f"基准坏账率: {total_bad_rate:.2%} | 样本: {n_total}")

    results = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_eval_single_rule)(r, df_values, col_index, label_arr, n_total, total_bad_rate, min_coverage, max_coverage, min_bads)
        for r in tqdm(valid_rules, desc="规则评估")
    )
    results = [r for r in results if r is not None]
    if not results:
        print("无满足条件的规则")
        return pd.DataFrame()
    result_df = pd.DataFrame(results).sort_values('bad_rate', ascending=False).reset_index(drop=True)
    result_df['bad_rate'] = result_df['bad_rate'].apply(lambda x: f"{x:.4f}")
    print(f"有效规则: {len(result_df)}")
    return result_df

In [ ]:
# ============================================================
# 执行规则评估
# ============================================================
good_vars = [
    "bh_br_scoreysstd", "bh_xys_byf_linglong_score82", "jbx_lhjm_v1_acard_score",
    "jbx_scorec_score_c", "pd_bwtj_score_v5_1", "pd_rong360_acard_dz_score_v2",
    "pd_tc_puchen_score", "pd_td_td111_score", "pd_yr_lm_score",
    "td_large_cash_score", "xys_total_qarnet_score46", "xysl_lanyu_score1",
    "yd_al_xiaoniu_scorea1", "zzx_pboc2_score", "pd_rong360_acard_dz_score",
    "tcxy_creditpro_v1_credit_model_score_high", "pd_dxm_dongzhi_score_v1",
    "pd_mayi_aft_v3_score",
]
bad_vars = [
    "baidu_panshi_multiloans_score", "bh_aly_anti_fraud_v6_score",
    "bh_hp_credit_operat_score", "bh_hp_custom_credit_score_y",
    "bwjk_relationship_network_score", "pd_bdy_finance_fraud_score",
    "pd_haluo_insightv7_score", "rong360_fraud_riskscore", "td_fraud_score",
    "txty_total_de4_score", "txty_total_xe4_5_score", "pd_rong360_acard_dz_score_p",
]

train_df = pd.concat([train_data_x, train_data_y], axis=1)
rule_report = evaluate_rules(
    train_df, top_rules.index, 'dob4_ever10_flg',
    good_vars=good_vars, bad_vars=bad_vars,
    min_coverage=0.10, max_coverage=0.50, min_bads=50, n_jobs=-1
)
rule_report.head(20)

---
# 评分卡建模部分

In [ ]:
# ============================================================
# 第十二部分：PSI稳定性计算
# ============================================================
def calculate_feature_psi(df, month_col, feature_list, base_month=None, bins=10):
    """计算各特征各月PSI"""
    if base_month is None:
        base_month = df[month_col].min()
        print(f"基准月: {base_month}")
    base_data = df[df[month_col] == base_month]
    psi_results = []

    for month in df[month_col].unique():
        if month == base_month:
            continue
        current_data = df[df[month_col] == month]
        for var in feature_list:
            if not pd.api.types.is_numeric_dtype(base_data[var]):
                continue
            exp = base_data[var].dropna()
            act = current_data[var].dropna()
            if len(exp) == 0 or len(act) == 0:
                continue
            breakpoints = np.percentile(exp, np.linspace(0, 100, bins+1))
            breakpoints[-1] += 1e-6
            exp_hist = np.histogram(exp, bins=breakpoints)[0] + 1e-6
            act_hist = np.histogram(act, bins=breakpoints)[0] + 1e-6
            exp_pct = exp_hist / exp_hist.sum()
            act_pct = act_hist / act_hist.sum()
            psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
            psi_results.append({'month': month, 'variable': var, 'psi': psi})

    return pd.DataFrame(psi_results)

# 执行PSI
pboc_feature = list(importance_df[importance_df.importance > 0].feature[:1000])
psi_df = calculate_feature_psi(
    df[(df['is_perform_dob2_ever30'] > 0) & (df['trans_month'] >= '202501')].copy(),
    month_col='trans_month', feature_list=pboc_feature, bins=5
)
psi_df.head()

In [ ]:
# 筛选PSI < 0.1的稳定变量
stable_vars = psi_df.groupby('variable')['psi'].max().reset_index()
stable_vars = stable_vars[stable_vars['psi'] < 0.1]['variable'].tolist()
print(f"稳定变量数: {len(stable_vars)}")

In [ ]:
# ============================================================
# 第十三部分：风险趋势稳定性
# ============================================================
def check_trend_stability(pivot_table, threshold=0.4):
    monthly_rankings = {m: pivot_table[m].rank(method='dense').values for m in pivot_table.columns}
    months = list(monthly_rankings.keys())
    taus = [kendalltau(monthly_rankings[months[i]], monthly_rankings[months[j]])[0]
            for i in range(len(months)) for j in range(i+1, len(months))]
    avg_tau = np.mean(taus)
    return {'avg_similarity': avg_tau, 'is_stable': avg_tau >= threshold}

def assess_stability(input_data, features, target, month_col,
                     n_bins=5, min_months=3, stability_thresh=0.4):
    data = input_data.copy()
    results = []
    for feature in features:
        try:
            if data[feature].nunique() > n_bins:
                binner = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
                data['bin'] = binner.fit_transform(data[[feature]])
            else:
                data['bin'] = data[feature]
            pivot = data.groupby(['bin', month_col])[target].mean().unstack().dropna(thresh=min_months, axis=1)
            if pivot.shape[1] < min_months:
                continue
            stability = check_trend_stability(pivot, threshold=stability_thresh)
            results.append({'feature': feature, **stability})
        except:
            pass
    return pd.DataFrame(results)

# 执行稳定性分析
input_data = df[(df.apply_month >= '202501') & (df.is_perform_dob1_ever30 == 1)].fillna(-90)

df_summary3 = assess_stability(input_data, stable_vars, 'dob1_ever30_flg', 'apply_month',
                               n_bins=3, min_months=2, stability_thresh=0.7)
features_3bins = list(df_summary3[df_summary3.is_stable == True].feature)
print(f"3分箱稳定特征: {len(features_3bins)}")

df_summary2 = assess_stability(input_data, stable_vars, 'dob1_ever30_flg', 'apply_month',
                               n_bins=2, min_months=2, stability_thresh=0.7)
features_2bins = [c for c in df_summary2[df_summary2.is_stable == True].feature.tolist()
                  if c not in features_3bins]
print(f"2分箱稳定特征: {len(features_2bins)}")

In [ ]:
# ============================================================
# 第十四部分：相关性分析与剔除
# ============================================================
same_trend_vars = list(set(features_3bins + features_2bins))
print(f"趋势稳定变量总数: {len(same_trend_vars)}")

def refine_stable_vars_with_importance(df, stable_vars, importance_df, threshold=0.5):
    """剔除高相关性变量（保留重要性高的）"""
    valid_vars = [v for v in stable_vars if v in df.columns]
    corr = df[valid_vars].corr().abs()
    imp = importance_df.set_index('feature')['importance']
    drop = set()
    mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
    for i, j in zip(*np.where(mask & (corr > threshold))):
        a, b = corr.columns[i], corr.columns[j]
        if a in imp.index and b in imp.index:
            drop.add(a if imp[a] < imp[b] else b)
    keep = [v for v in stable_vars if v not in drop]
    return importance_df[importance_df['feature'].isin(keep)].sort_values('importance', ascending=False).reset_index(drop=True)

final_imp_df = refine_stable_vars_with_importance(df, same_trend_vars, importance_df, 0.6)
print(f"去相关后特征数: {final_imp_df.shape[0]}")
print(list(final_imp_df.feature))

In [ ]:
# ============================================================
# 第十五部分：逻辑回归评分卡训练
# ============================================================
ncols = 100
categorical_variables = []

# 需要额外排除的变量（根据业务判断）
drop_vars = [
    'br_sx_als_m6_cell_bank_max_monnum', 'model_jdzad_zaac2v020_a0t630x0_score',
    'model_market_sasg00v40_5lx_pboc2_nifa_score', 'pboc2_micro_loan_query_cnt_3m',
    'pboc2_open_cd_used_limit_6m_avg_rate', 'tcxy_creditpro_v1_fundsfail_cnt_tot_ratio_180d',
    'tcxy_creditpro_v1_rpysuc_atsm_360d_rt', 'tcxy_creditpro_v1_guarantee_trans_cur_diffdays_min',
    'model_acard_zaac00v13_7m4x_score', 'model_jdzad_acard_zaac00v90_9lx_score',
    'xys_total_qarnet_score46', 'model_jdzad_acard_bhdj_v2_score',
    'tcxy_creditpro_v1_last5repaysuc_cntratio_by_repaycnttotal',
    'tcxy_creditpro_v1_last5trans_fundsfail_tot_ratio',
]

cols = sorted(list(set([x for x in final_imp_df.feature[:ncols] if x not in drop_vars] + categorical_variables)))

features_3 = [x for x in cols if x in features_3bins + categorical_variables]
features_2 = [x for x in cols if x not in features_3]
binning_config = {**{f: {"max_n_bins": 3} for f in features_3},
                  **{f: {"max_n_bins": 2} for f in features_2}}
print(f'入模特征数: {len(cols)}')

# 训练数据
df_train = df[df.is_perform_dob3_ever30 == 1]
print(f'训练样本: {df_train.shape}')
X = df_train[cols]
y = df_train.dob3_ever30_flg

# 分箱+评分卡
binning_process = BinningProcess(
    variable_names=X.columns.tolist(),
    categorical_variables=categorical_variables,
    min_bin_size=0.15,
    binning_fit_params=binning_config
)

scorecard = Scorecard(
    binning_process=binning_process,
    estimator=LogisticRegression(random_state=42),
    scaling_method="pdo_odds",
    scaling_method_params={"pdo": 20, "odds": 50, "scorecard_points": 600}
)
scorecard.fit(X, y)

# 评分卡表
scorecard_table = scorecard.table(style="detailed")
scorecard_table[scorecard_table.Count > 0]

In [ ]:
# ============================================================
# 第十六部分：评分卡效果验证
# ============================================================
def validate_scorecard(df, scorecard, target_col, month_col='trans_month', months=None):
    """批量验证评分卡在不同月份/全量的分效果"""
    col = 'prediction'
    c = Combiner()

    # 全量
    df_all = df[df[month_col] >= '202501']
    pred = scorecard.score(df_all)
    dfpred = pd.DataFrame({col: pred, target_col: df_all[target_col]})
    c.fit(dfpred[[col, target_col]], y=target_col, method='quantile', n_bins=10, empty_separate=True)
    print("=== 全量 ===")
    bin_plot(c.transform(dfpred[[col, target_col]], labels=True), x=col, target=target_col,
             annotate_format=".3f", figsize=(8, 5))

    # 分月
    if months is None:
        months = sorted(df[month_col].unique())[-4:]
    for m in months:
        df_m = df[df[month_col] == m]
        if len(df_m) < 100:
            continue
        pred = scorecard.score(df_m)
        dfpred = pd.DataFrame({col: pred, target_col: df_m[target_col]})
        c.fit(dfpred[[col, target_col]], y=target_col, method='quantile', n_bins=10, empty_separate=True)
        print(f"\n=== {m} ===")
        bin_plot(c.transform(dfpred[[col, target_col]], labels=True), x=col, target=target_col,
                 annotate_format=".3f", figsize=(8, 5))

# dob3_ever30验证
validate_scorecard(df, scorecard, 'dob3_ever30_flg', months=['202501','202502','202503','202504'])

# fpd10验证
validate_scorecard(df, scorecard, 'fpd10_flag', months=['202501','202502','202503','202504'])

In [ ]:
# ============================================================
# 第十七部分：保存模型
# ============================================================
joblib.dump(scorecard, '/root/策略/oppo/model_oppo_dob2_ever30_001.pkl')
print("模型已保存")